## Imports + EDA

In [ ]:
# Imports
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats
import warnings
from statsmodels.graphics.tsaplots import plot_pacf
import matplotlib.pyplot as plt

In [ ]:
BASE_URL = "https://raw.githubusercontent.com/siddharthyashkukreja-cloud/QuantInvesting/35528ac1dde5d2d873e1adfd0864381326d3b768/"

# Load the data
dfPredictors = pd.read_csv(f"{BASE_URL}/predictor_data.csv")
dfPredictors.drop(columns=['Unnamed: 0'], axis=1, inplace=True) # Drop unnecessary column

variables_to_negate = ['ntis', 'tbl', 'lty', 'lagINFL']
for var in variables_to_negate:
    dfPredictors[var] = -dfPredictors[var]
dfPredictors.head()

In [ ]:

# Determine number of columns (excluding yyyyq which is our x-axis)
num_cols = len(dfPredictors.columns) - 1
num_rows = (num_cols + 2) // 3  # 3 plots per row, rounded up

# Create figure with subplots
fig, axes = plt.subplots(num_rows, 3, figsize=(18, num_rows * 4), sharex=True)
axes = axes.flatten()  # Flatten to make indexing easier

# Plot each column in its own subplot
col_idx = 0
for col in dfPredictors.columns:
    if col != 'yyyyq':  # Skip the x-axis column
        axes[col_idx].plot(dfPredictors['yyyyq'], dfPredictors[col])
        axes[col_idx].set_title(col)
        axes[col_idx].grid(True)
        col_idx += 1

# Hide any unused subplots
for i in range(col_idx, len(axes)):
    fig.delaxes(axes[i])

# Set major ticks at reasonable intervals for better readability
tick_indices = np.linspace(0, len(dfPredictors)-1, 6, dtype=int)
tick_values = dfPredictors.iloc[tick_indices]['yyyyq'].values

# Apply x-ticks to all subplots
for ax in axes[:col_idx]:
    ax.set_xticks(tick_values)
    ax.set_xticklabels(tick_values, rotation=45)

# Add overall title and x-axis label
plt.suptitle('Financial Predictors Trends Over Time', fontsize=16)
fig.text(0.5, 0.04, 'Year-Quarter (yyyyq)', ha='center', fontsize=12)

# Tight layout
plt.tight_layout()
plt.subplots_adjust(top=0.95, bottom=0.1)
plt.show()

In [ ]:
# Calculate the correlation matrix
correlation_matrix = dfPredictors[dfPredictors.columns[1:]].corr() # Exclude 'yyyyq' from correlation calculation

# Display the correlation matrix
print("Correlation Matrix:")
correlation_matrix

# Plot the correlation matrix as a heatmap
plt.figure(figsize=(14, 12))
im = plt.imshow(correlation_matrix, cmap='coolwarm')
plt.colorbar(im)
plt.xticks(range(len(correlation_matrix.columns)), correlation_matrix.columns, rotation=90)
plt.yticks(range(len(correlation_matrix.columns)), correlation_matrix.columns)
plt.title('Correlation Matrix Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
### PACF test for Lag Selection

lags = 20  
plot_pacf(dfPredictors['r'], lags=lags, method='ywm', alpha=0.05)
plt.title("Partial Autocorrelation Function (PACF) for 'r'")
plt.xlabel("Lags")
plt.ylabel("PACF")
plt.grid()
plt.show()


## Question 1 simple regression for each predictor

In [ ]:

def simple_regression(data, dependant_var, independent_var):
    """Performs a simple linear regression of dependant_var on independent_var.
    Args:
        pdData: The dataframe containing the data.
        strDependant_var : The name of the dependent variable column.
        strIndependent_var : The name of the independent variable column.

    Returns:
        dict: A dictionary containing regression results, coefficients, adjusted R², p-values, and robust standard errors in a list
    """
    X = data[independent_var][:-1].reset_index(drop=True) 
    y = data[dependant_var][1:].reset_index(drop=True)


    X = sm.add_constant(X)  # Adds a constant term to the predictor
    model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 4})  # Using robust standard errors


    t_stat = model.tvalues.iloc[1]  # t-statistic for the independent variable
    two_sided_p = model.pvalues[independent_var]

    if t_stat > 0:
        one_sided_p = two_sided_p / 2
    else:
        one_sided_p = 1 - (two_sided_p / 2)

    # Extract relevant information
    results = {
        "predicted_name": f"{independent_var}", # get the string name of the predictor
        "beta": model.params.iloc[1],
        "HAC t_tests": t_stat,
        "one_sided_pvalue": one_sided_p,  # One-tailed p-value
        "Adj. R²": model.rsquared_adj,
    }

    return results

# Make a list containing dictionaries of regression results for each predictor
all_results = []

for col in dfPredictors.columns[2:]:
    result = simple_regression(dfPredictors, 'r', col)
    all_results.append(result)

indiv_regression_results = pd.DataFrame(all_results)
indiv_regression_results



## Question 2

In [ ]:
# Splits of the data
total_obs = len(dfPredictors)
m = 80 # Initial in-sample period (20 years * 4 quarters)
p = 40 #hold out period (10 years * 4 quarters)
q = total_obs - m - p  # Out-of-sample period

# --- BENCHMARK FORECASTS ---

# Basic mean for each window of data (AKA historical benchmark)
benchmark_forecasts = []
for t in range(m + p, total_obs):
    # Benchmark is the expanding mean of data up to period t-1
    historical_mean = dfPredictors['r'].iloc[0:t].mean()
    benchmark_forecasts.append(historical_mean)

# Get the actual returns for the correct OOS period
actual_returns_oos = dfPredictors['r'].iloc[m + p : total_obs].values
mspe_benchmark = np.mean((actual_returns_oos - np.array(benchmark_forecasts))**2)

oos_results_list = []
predictor_variables = dfPredictors.columns[2:]

# These dictionaries will store the forecasts to be used in Question 4
holdout_forecasts_dict = {}
oos_forecasts_dict = {}

for predictor in predictor_variables:
    model_forecasts = []
    
    # Loop from the end of the in-sample period (m-1) to the second-to-last period
    for t in range(m + p, total_obs):
        # Esimation window from starts at 0 and expands from m+p to total_obs
        df_window = dfPredictors.iloc[0:t]
        
        y = df_window['r']
        X = df_window[predictor].shift(1)
        X = sm.add_constant(X)
        
        model = sm.OLS(y, X, missing='drop').fit()
        
        # Forecast for period t using predictor value at t-1
        last_predictor_value = df_window[predictor].iloc[-1]
        forecast = model.params['const'] + model.params[predictor] * last_predictor_value
        
        model_forecasts.append(forecast)

    mspe_model = np.mean((actual_returns_oos - np.array(model_forecasts))**2)
    r2_oos = 1 - (mspe_model / mspe_benchmark)
    
    oos_results_list.append({
        'Predictor': predictor,
        'R2_OOS': r2_oos,
        'Outperforms': r2_oos > 0
    })  

# Display results of each regression on predictor variable
oos_results_df = pd.DataFrame(oos_results_list)
print("\n--- Corrected Out-of-Sample R-squared Results ---")
print(oos_results_df.sort_values('R2_OOS', ascending=False).to_string(index=False))

## Question 3 (Kitchen Sink forecast)

In [ ]:
predictor_variables = dfPredictors.columns[2:]

# List to store the forecasts from the kitchen sink model
kitchen_sink_forecasts = []

# Loop from the end of the in-sample period to the second-to-last observation
for t in range(m+p+1, total_obs -1):
    df_window = dfPredictors.iloc[0:t+1]

    y = df_window['r']
    X = df_window[predictor_variables].shift(1) 
    X = sm.add_constant(X)
    
    model = sm.OLS(y, X, missing='drop').fit()
    
    last_predictor_values = df_window[predictor_variables].iloc[-1].tolist()

    # 
    X_to_predict = [1.0] + last_predictor_values

    # Use this array to make the forecast
    forecast = model.predict(X_to_predict)[0]

    kitchen_sink_forecasts.append(forecast)

# Compare kitchen sink model to benchmark
actual_returns_ks = dfPredictors['r'].iloc[m+p+1+1 : total_obs].values
mspe_kitchen_sink = np.mean((actual_returns_ks - np.array(kitchen_sink_forecasts))**2)

r2_oos_kitchen_sink = 1 - (mspe_kitchen_sink / mspe_benchmark)

# --- Display the final result for Question 3 ---
print(f"\nKitchen Sink Model R2_OOS: {r2_oos_kitchen_sink:.6f}")
if r2_oos_kitchen_sink > 0:
    print("The Kitchen Sink model outperforms the historical mean benchmark.")
else:
    print("The Kitchen Sink model does NOT outperform the historical mean benchmark.")

#### New cell for question 2&3 that stores values for question 4 for the holdout period and out of sample (NOT FUNCTIONAL, Problem with indexing of actual return I believe)

In [ ]:
Y_full = dfPredictors['r'].values
# These dictionaries will store the forecasts to be used in Question 4
holdout_forecasts_dict = {}
oos_forecasts_dict = {}

for predictor in predictor_variables:
    X_full = dfPredictors[predictor].values
    
    # Initialize lists to store forecasts for the current predictor
    holdout_forecasts = []
    oos_forecasts = []
    
    # Generate forecasts for the HOLDOUT period (p)
    for t in range(m, m + p):
        df_window = dfPredictors.iloc[0:t+1]
        y, X = df_window['r'], sm.add_constant(df_window[predictor].shift(1))
        model = sm.OLS(y, X, missing='drop').fit()
        forecast = model.predict([1, df_window[predictor].iloc[-1]])[0]
        # Append to the temporary list for holdout forecasts
        holdout_forecasts.append(forecast)
        
    # Generate forecasts for the OUT-OF-SAMPLE period (q)
    for t in range(m + p, total_obs - 1):
        df_window = dfPredictors.iloc[0:t+1]
        y, X = df_window['r'], sm.add_constant(df_window[predictor].shift(1))
        model = sm.OLS(y, X, missing='drop').fit()
        forecast = model.predict([1, df_window[predictor].iloc[-1]])[0]
        # Append to the temporary list for final OOS forecasts
        oos_forecasts.append(forecast)
        
    # Store all the forecast for both periods in the dictionaries for question 4
    holdout_forecasts_dict[predictor] = np.array(holdout_forecasts)
    oos_forecasts_dict[predictor] = np.array(oos_forecasts)

print("All individual forecasts have been generated and stored for question 4")

# Benchmark for the final OOS period (q)
benchmark_forecasts_oos = [Y_full[0:t+1].mean() for t in range(m + p, total_obs - 1)]
actual_returns_oos = Y_full[m + p + 1 : total_obs]
mspe_benchmark_oos = np.mean((actual_returns_oos - np.array(benchmark_forecasts_oos))**2)

oos_r2_results = []
for predictor in predictor_variables:
    mspe_model = np.mean((actual_returns_oos - oos_forecasts_dict[predictor])**2)
    r2_oos = 1 - (mspe_model / mspe_benchmark_oos)
    oos_r2_results.append({'Predictor': predictor, 'R2_OOS': r2_oos})

oos_df = pd.DataFrame(oos_r2_results)
print("\n--- Question 2: Individual Predictor Out-of-Sample R-squared Results ---")
print(oos_df.sort_values('R2_OOS', ascending=False).to_string(index=False))


# Kitchen Sink Model for Question 3 
kitchen_sink_forecasts = []
for t in range(m + p, total_obs - 1):
    df_window = dfPredictors.iloc[0:t+1]
    y = df_window['r']
    X = sm.add_constant(df_window[predictor_variables].shift(1))
    model = sm.OLS(y, X, missing='drop').fit()
    X_to_predict = [1.0] + df_window[predictor_variables].iloc[-1].tolist()
    forecast = model.predict(X_to_predict)[0]
    kitchen_sink_forecasts.append(forecast)

mspe_kitchen_sink = np.mean((actual_returns_oos - np.array(kitchen_sink_forecasts))**2)
r2_oos_kitchen_sink = 1 - (mspe_kitchen_sink / mspe_benchmark_oos)
print(f"\n--- Question 3: Kitchen Sink Model R2_OOS ---")
print(f"R2_OOS: {r2_oos_kitchen_sink:.6f}")

## Question 4


In [ ]:
# Dictionaries: holdout_forecasts_dict, oos_forecasts_dict
# Numpy Arrays: actual_returns_oos, mspe_benchmark_oos
# =============================================================================

# --- 1. Mean and Median Combinations ---
# Create a matrix of individual forecasts (models x time)
individual_forecasts_matrix = np.array([oos_forecasts_dict[p] for p in predictor_variables])

# Calculate mean and median across models (axis=0) for each time period
mean_combo_forecasts = individual_forecasts_matrix.mean(axis=0)
median_combo_forecasts = np.median(individual_forecasts_matrix, axis=0)

# Evaluate Mean Combination
mspe_mean_combo = np.mean((actual_returns_oos - mean_combo_forecasts)**2)
r2_oos_mean_combo = 1 - (mspe_mean_combo / mspe_benchmark_oos)

# Evaluate Median Combination
mspe_median_combo = np.mean((actual_returns_oos - median_combo_forecasts)**2)
r2_oos_median_combo = 1 - (mspe_median_combo / mspe_benchmark_oos)

# --- 2. DMSPE Combination ---
def calculate_dmspe_forecasts(theta):
    """Calculates DMSPE weights from holdout period and applies them to OOS forecasts."""
    phi_weights = []
    actual_returns_holdout = Y_full[m + 1 : m + p + 1]
    
    # Step 1: Calculate weights based on holdout period performance
    for predictor in predictor_variables:
        forecasts_holdout = holdout_forecasts_dict[predictor]
        errors_sq = (actual_returns_holdout - forecasts_holdout)**2
        
        # Calculate discounted sum of squared errors (phi)
        phi = sum([(theta**(p - 1 - s)) * errors_sq[s] for s in range(p)])
        phi_weights.append(1 / (phi + 1e-8)) # Add small constant for stability
        
    # Normalize weights to sum to 1
    phi_weights = np.array(phi_weights) / np.sum(np.array(phi_weights))
    
    # Step 2: Apply fixed weights to the out-of-sample forecasts
    dmspe_combo_forecasts = np.dot(phi_weights, individual_forecasts_matrix)
    
    return dmspe_combo_forecasts

# Calculate DMSPE for theta = 0.9
dmspe_forecasts_09 = calculate_dmspe_forecasts(theta=0.9)
mspe_dmspe_09 = np.mean((actual_returns_oos - dmspe_forecasts_09)**2)
r2_oos_dmspe_09 = 1 - (mspe_dmspe_09 / mspe_benchmark_oos)

# Calculate DMSPE for theta = 1.0
dmspe_forecasts_10 = calculate_dmspe_forecasts(theta=1.0)
mspe_dmspe_10 = np.mean((actual_returns_oos - dmspe_forecasts_10)**2)
r2_oos_dmspe_10 = 1 - (mspe_dmspe_10 / mspe_benchmark_oos)

# Calculate DMSPE for theta = 0.5
dmspe_forecasts_05 = calculate_dmspe_forecasts(theta=0.5)
mspe_dmspe_05 = np.mean((actual_returns_oos - dmspe_forecasts_05)**2)
r2_oos_dmspe_05 = 1 - (mspe_dmspe_05 / mspe_benchmark_oos)

# Calculate DMSPE for theta = 0.2
dmspe_forecasts_02 = calculate_dmspe_forecasts(theta=0.2)
mspe_dmspe_02 = np.mean((actual_returns_oos - dmspe_forecasts_02)**2)
r2_oos_dmspe_02 = 1 - (mspe_dmspe_02 / mspe_benchmark_oos)

# Calculate DMSPE for theta = 0.1
dmspe_forecasts_01 = calculate_dmspe_forecasts(theta=0.1)
mspe_dmspe_01 = np.mean((actual_returns_oos - dmspe_forecasts_01)**2)
r2_oos_dmspe_01 = 1 - (mspe_dmspe_01 / mspe_benchmark_oos)

# Calculate DMSPE for theta = 0.01
dmspe_forecasts_001 = calculate_dmspe_forecasts(theta=0.01)
mspe_dmspe_001 = np.mean((actual_returns_oos - dmspe_forecasts_001)**2)
r2_oos_dmspe_001 = 1 - (mspe_dmspe_001 / mspe_benchmark_oos)

# --- 3. Display Final Results for Question 4 ---
print("\n--- Forecast Combination R2_OOS Results ---")
print(f"Mean Combination:     {r2_oos_mean_combo:.6f}")
print(f"Median Combination:   {r2_oos_median_combo:.6f}")
print(f"DMSPE (theta=0.9):    {r2_oos_dmspe_09:.6f}")
print(f"DMSPE (theta=1.0):    {r2_oos_dmspe_10:.6f}")
print(f"DMSPE (theta=0.5):    {r2_oos_dmspe_05:.6f}")
print(f"DMSPE (theta=0.2):    {r2_oos_dmspe_02:.6f}")
print(f"DMSPE (theta=0.1):    {r2_oos_dmspe_01:.6f}")
print(f"DMSPE (theta=0.01):   {r2_oos_dmspe_001:.6f}")

# Plot R²_OOS results for different levels of theta
theta_values = [0.01,0.1, 0.2, 0.5, 0.9, 1.0]
r2_oos_dmspe_values = [r2_oos_dmspe_001, r2_oos_dmspe_01, r2_oos_dmspe_02, r2_oos_dmspe_05, r2_oos_dmspe_09, r2_oos_dmspe_10]
plt.figure(figsize=(10, 6))
plt.plot(theta_values, r2_oos_dmspe_values, marker='o')
plt.title('R²_OOS vs Theta in DMSPE Combination')
plt.xlabel('Theta')
plt.ylabel('R²_OOS')
plt.grid()
plt.xticks(theta_values)
plt.show()

## Question 5